## MAI-Voice-2-Preview: Multilingual Prompted Text-to-Speech

**Model card reference:** MAI-Voice-2 (Foundry) latest update

MAI-Voice-2-Preview is a high-fidelity, expressive, prompted TTS model across 15 languages and 18 locales.
This notebook demonstrates Foundry-oriented Speech SDK patterns (Entra auth with resource ID + optional key auth), multilingual synthesis, and practical implementation notes.


## 1. Setup


### Environment variables

| Variable | Required | Secret | Purpose |
|---|---|---|---|
| `MAI_VOICE_2_ENDPOINT` | Optional | No | Foundry/Speech endpoint (for example `https://<resource>.cognitiveservices.azure.com/`). |
| `MAI_VOICE_2_RESOURCE_ID` | Conditional | No | ARM resource ID required when `USE_ENTRA_AUTH=true` for `aad#<resource-id>#<token>` authorization. |
| `MAI_VOICE_2_KEY` | Conditional | **Yes** | API key when `USE_ENTRA_AUTH=false`. |
| `USE_ENTRA_AUTH` | Optional | No | Set `true` (default) to use Entra auth, `false` to force key auth. |
| `MAI_VOICE_2_OUTPUT_DIR` | Optional | No | Output directory for generated audio; defaults to `media/mai-voice-2`. |

`MAI_VOICE_2_RESOURCE_ID` is required when `USE_ENTRA_AUTH=true`; `MAI_VOICE_2_KEY` is required when `USE_ENTRA_AUTH=false`.

Do not commit `.env` or `deployment.env` files with secrets.


In [ ]:
# %pip install -q azure-cognitiveservices-speech python-dotenv azure-identity


In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
import azure.cognitiveservices.speech as speechsdk
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

ENV_PATH = 'deployment.env' if os.path.exists('deployment.env') else os.path.join('..', 'deployment.env')
load_dotenv(ENV_PATH, override=True)

VOICE2_ENDPOINT = (
    os.getenv('MAI_VOICE_2_ENDPOINT')
    or os.getenv('VOICE_SPEECH_ENDPOINT')
    or 'https://voiceliveapipoc.cognitiveservices.azure.com/'
).rstrip('/')
VOICE2_KEY = (
    os.getenv('MAI_VOICE_2_KEY')
    or os.getenv('VOICE_SPEECH_KEY')
    or os.getenv('AZURE_SPEECH_KEY')
)
VOICE2_RESOURCE_ID = os.getenv('MAI_VOICE_2_RESOURCE_ID', '').strip()
USE_ENTRA_AUTH = os.getenv('USE_ENTRA_AUTH', 'true').lower() == 'true'

voice_output_env = os.getenv('MAI_VOICE_2_OUTPUT_DIR')
OUT_DIR = Path(voice_output_env) if voice_output_env else Path('media') / 'mai-voice-2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

parsed = urlparse(VOICE2_ENDPOINT)
VOICE2_BASE_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
credential = DefaultAzureCredential()

if USE_ENTRA_AUTH:
    assert VOICE2_RESOURCE_ID, 'Set MAI_VOICE_2_RESOURCE_ID when USE_ENTRA_AUTH=true'
else:
    assert VOICE2_KEY, 'Set MAI_VOICE_2_KEY when USE_ENTRA_AUTH=false'

def build_speech_config(voice_name: str) -> speechsdk.SpeechConfig:
    if USE_ENTRA_AUTH:
        token = credential.get_token('https://cognitiveservices.azure.com/.default')
        config = speechsdk.SpeechConfig(endpoint=VOICE2_BASE_ENDPOINT)
        config.authorization_token = f"aad#{VOICE2_RESOURCE_ID}#{token.token}"
    else:
        config = speechsdk.SpeechConfig(subscription=VOICE2_KEY, endpoint=VOICE2_BASE_ENDPOINT)

    config.speech_synthesis_voice_name = voice_name
    config.set_speech_synthesis_output_format(
        speechsdk.SpeechSynthesisOutputFormat.Audio24Khz160KBitRateMonoMp3
    )
    return config

print(f'Endpoint: {VOICE2_BASE_ENDPOINT}')
print(f"Auth mode: {'Entra ID' if USE_ENTRA_AUTH else 'API key'}")
print('Default sample voice: en-US-Harper:MAI-Voice-2-Preview')
print('Output target: 24kHz MP3')


## 2. Model Card Highlights


- High-fidelity natural voice synthesis with expressive control.
- Generate speech from short audio prompts (5-60 seconds).
- Multilingual support across **15 languages and 18 locales**.
- Supported languages: Arabic, Chinese, English, French, German, Hindi, Indonesian, Italian, Japanese, Korean, Portuguese, Russian, Spanish, Thai, and Vietnamese.
- Supports long-form content generation via chunking with context carryover.
- Output format is 24kHz mono audio.
- Served globally via East US, Sweden Central, and Southeast Asia.
- Pricing reference: **$22 per 1M characters**.
- Out-of-scope note: optimized for naturalness/expressivity over ultra-low-latency scenarios.


## 3. Reference Foundry SDK Pattern


In [ ]:
reference_voice = 'de-DE-Klaus:MAI-Voice-2-Preview'
reference_text = 'Hello, welcome to Azure AI Foundry!'

RUN_REFERENCE_CALL = False
if RUN_REFERENCE_CALL:
    reference_config = build_speech_config(reference_voice)
    reference_synth = speechsdk.SpeechSynthesizer(speech_config=reference_config)
    result = reference_synth.speak_text_async(reference_text).get()
    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print(f'Speech synthesized for text [{reference_text}]')
    elif result.reason == speechsdk.ResultReason.Canceled:
        details = result.cancellation_details
        print(f'Speech synthesis canceled: {details.reason}')
        if details.reason == speechsdk.CancellationReason.Error:
            print(f'Error details: {details.error_details}')
else:
    print('Set RUN_REFERENCE_CALL=True to execute the Foundry SDK sample.')
    print('Endpoint:', VOICE2_BASE_ENDPOINT)
    print('Voice:', reference_voice)
    print('Auth mode:', 'Entra ID (resource ID token)' if USE_ENTRA_AUTH else 'API key')


## 4. Helper: Synthesize Text to File


In [ ]:
def synthesize_text_to_file(text: str, voice_name: str, out_file: str) -> Path:
    output_path = OUT_DIR / out_file
    audio_config = speechsdk.audio.AudioOutputConfig(filename=str(output_path))
    speech_config = build_speech_config(voice_name)
    synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)
    result = synthesizer.speak_text_async(text).get()

    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print(f'Wrote {output_path} ({output_path.stat().st_size:,} bytes)')
        return output_path

    details = result.cancellation_details
    if details.reason == speechsdk.CancellationReason.Error:
        raise RuntimeError(f'Speech synthesis failed: {details.error_details}')
    raise RuntimeError(f'Speech synthesis canceled: {details.reason}')


## 5. Multilingual Synthesis Samples


Illustrative sample audio from one run:

<audio controls src="media/mai-voice-2/01-mai-voice2-en.mp3"></audio>

- [Download sample (`01-mai-voice2-en.mp3`)](media/mai-voice-2/01-mai-voice2-en.mp3)

Audio style and prosody can vary between runs and model updates.


In [ ]:
samples = [
    {'voice': 'en-US-Harper:MAI-Voice-2-Preview', 'text': 'Hello from MAI Voice 2 in English.', 'out': 'mai_voice2_en.mp3'},
    {'voice': 'es-MX-Valeria:MAI-Voice-2-Preview', 'text': 'Hola, esta es una muestra de MAI Voice 2.', 'out': 'mai_voice2_es.mp3'},
    {'voice': 'fr-FR-Soleil:MAI-Voice-2-Preview', 'text': 'Bonjour, ceci est un exemple MAI Voice 2.', 'out': 'mai_voice2_fr.mp3'},
    {'voice': 'de-DE-Klaus:MAI-Voice-2-Preview', 'text': 'Hallo, dies ist eine MAI Voice 2 Probe.', 'out': 'mai_voice2_de.mp3'},
]

for s in samples:
    try:
        synthesize_text_to_file(s['text'], s['voice'], s['out'])
    except Exception as ex:
        print(f"{s['voice']} failed: {ex}")


## 6. Voice Prompting Note (Gated Access)


Voice prompting (personal voice cloning) is gated and requires Microsoft approval plus consent safeguards.

Implementation reminders from the model card:
1. Apply for limited access approval.
2. Upload consent audio + prompt.
3. Use Personal Voice APIs to create voice profile.
4. Synthesize with approved voice profile.


## 7. Next Steps

1. Set MAI_VOICE_2_PRICE_PER_1M_CHAR after MAI-Voice-2 pricing is published.
2. Replace sample voices with the final published MAI-Voice-2 voice list.
3. Add latency benchmarking if your scenario is latency-sensitive.
